In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        # TODO 1: Слой Multi-Head Attention.
        # Укажи нужные параметры и обязательно batch_first=True.
        self.mha = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

        # TODO 2: Два ОТДЕЛЬНЫХ слоя LayerNorm (на вход принимают embed_dim).
        # Один для первого блока (Add & Norm 1), другой для второго (Add & Norm 2).
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)


        # TODO 3: Feed-Forward Network (FFN).
        # Создай два линейных слоя:
        self.linear1 = nn.Linear(embed_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, embed_dim)

        # TODO 4: Слой Dropout.
        # Будем использовать его после Attention и после FFN для регуляризации.
        self.dropout = nn.Dropout(dropout_rate)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Вход 'x' имеет форму [batch_size, seq_len, embed_dim]

        # ==========================================
        # БЛОК 1: Attention -> Dropout -> Add & Norm
        # ==========================================
        # TODO 5: Пропусти 'x' через MultiheadAttention (query, key и value равны 'x').
        # Забери только тензор с результатами (веса внимания нам сейчас не нужны).
        out, weights = self.mha(query=x, key=x, value=x)

        # TODO 6: Пропусти результат внимания через слой Dropout.
        out = self.dropout(out)

        # TODO 7: Сделай Residual Connection (сложи оригинальный 'x' и результат шага 6).
        out = x + out

        # TODO 8: Пропусти полученную сумму через ПЕРВЫЙ слой LayerNorm.
        # Сохрани результат в переменную (например, 'out1').
        out1 = self.ln1(out)

        # ==========================================
        # БЛОК 2: FFN -> Активация -> Dropout -> Add & Norm
        # ==========================================
        # TODO 9: Пропусти 'out1' через первый линейный слой FFN.
        out1 = self.linear1(out1)
        
        # TODO 10: Примени функцию активации (например, F.relu).
        out1 = self.relu(out1)

        # TODO 11: Пропусти результат через второй линейный слой FFN.
        out1 = self.linear2(out1)

        # TODO 12: Пропусти результат шага 11 через Dropout.
        out1 = self.dropout(out1)

        # TODO 13: Сделай второй Residual Connection (сложи 'out1' и результат шага 12).
        out1 = x + out1

        # TODO 14: Пропусти сумму через ВТОРОЙ слой LayerNorm. Верни итоговый тензор.
        out1 = self.ln2(out1)
        return out1

In [ ]:
# ==========================================
# ПРОВЕРКА КОДА
# ==========================================
# 1. Инициализируй блок (например: embed_dim=256, num_heads=8, ff_dim=1024)
model = TransformerEncoderBlock(embed_dim=256, num_heads=8, ff_dim=1024)

# 2. Создай фейковый батч данных: 3 текста по 10 слов, размер эмбеддинга 256
#    Используй torch.randn(...)
batch = torch.randn(3, 10, 256)

# 3. Пропусти батч через блок.
out = model(batch)

# 4. Распечатай размерность входа и размерность выхода. Они должны быть АБСОЛЮТНО одинаковыми.
print(batch.shape, out.shape)